In [1]:
#!pip install -U torch scikit-learn torchvision timm

# EfficientNet-B0 Vanilla Baseline + Targeted Tuning

- EfficientNet-B0, ImageNet pretrained (`timm`)
- Same 70/15/15 stratified split, seed, preprocessing, augmentation and class weighting
- Two-phase transfer learning: frozen classifier head → fine-tune final EfficientNet block
- Small validation-only hyperparameter sweep
- Winner selected by **lowest Phase-2 validation loss**
- Held-out test set evaluated only after tuning and final training are complete
- Saves frozen + fine-tuned metrics, histories, confusion matrices and checkpoints


In [2]:
import argparse
import copy
import json
import os
import random
from pathlib import Path
 
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report,confusion_matrix,roc_auc_score, accuracy_score

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from tqdm import tqdm
 
try:
    import timm
except ImportError as e:
    raise ImportError(
        "This script requires `timm`. Install with: pip install timm --break-system-packages"
    ) from e

## Reproducibility

In [3]:
def set_seed(seed: int = 42):
    """Set the random seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## Data

In [4]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

In [5]:
def build_transforms(img_size: int = 224):
  train_tf = transforms.Compose(
    [
      transforms.Resize((img_size, img_size)),
      transforms.RandomCrop(img_size, padding=8, padding_mode='reflect'),
      transforms.RandomHorizontalFlip(p=0.5),
      transforms.RandomRotation(degrees=10),
      transforms.ColorJitter(brightness=0.2, contrast=0.2),
      transforms.Grayscale(num_output_channels=3),
      transforms.ToTensor(),
      transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ]
  )

  eval_tf = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
  ])
  return train_tf, eval_tf

In [6]:
class TransformSubset(Dataset):
  """Wraps a Subset of an ImageFolder so train/val/test can each use a different transform."""

  def __init__(self, subset: Subset, transform):
    self.subset = subset
    self.transform = transform

  def __len__(self):
    return len(self.subset)

  def __getitem__(self, idx):
    img, label = self.subset[idx]
    if self.transform is not None:
      img = self.transform(img)
    return img, label

In [7]:
def stratified_split(dataset: ImageFolder, seed: int = 42):
  """70/15/15 stratified split of dataset into train, val, and test subsets."""

  targets = np.array(dataset.targets)
  indices = np.arange(len(dataset))

  train_idx, temp_idx = train_test_split(
    indices, test_size=0.3, stratify=targets, random_state=seed
  )

  val_idx, test_idx = train_test_split(
    temp_idx,
    test_size= 0.50,
    stratify=targets[temp_idx],
    random_state = seed,
  )

  return train_idx, val_idx, test_idx

In [8]:
def build_dataloaders(data_dir: str, img_size: int, batch_size: int, seed: int, num_workers: int = 4):
  # ImageFolder expects: data_dir/<class_name>/*.png
  # Load once without transform so PIL images can be transformed differently per split.
  def only_images_folder(path):
    p = Path(path)
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".gif"}
    return p.parent.name.lower() == "images" and p.suffix.lower() in valid_exts
  base_dataset = ImageFolder(root=data_dir, is_valid_file=only_images_folder)

  train_idx, val_idx, test_idx = stratified_split(base_dataset, seed=seed)
  train_tf, eval_tf = build_transforms(img_size)

  train_ds = TransformSubset(Subset(base_dataset, train_idx), train_tf)
  val_ds = TransformSubset(Subset(base_dataset, val_idx), eval_tf)
  test_ds = TransformSubset(Subset(base_dataset, test_idx), eval_tf)

  pin_memory = torch.cuda.is_available()
  train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
  val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
  test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

  class_names = base_dataset.classes
  train_targets = np.array(base_dataset.targets)[train_idx]
  datasets = {"train": train_ds, "val": val_ds, "test": test_ds}
  return train_loader, val_loader, test_loader, class_names, train_targets, datasets

In [9]:
def compute_class_weights(train_targets: np.ndarray, num_classes: int) -> torch.Tensor:
  counts = np.bincount(train_targets, minlength=num_classes).astype(np.float32)
  counts[counts == 0] = 1.0  # avoid div-by-zero
  weights = counts.sum() / (num_classes * counts)
  return torch.tensor(weights, dtype=torch.float32)

## Model

In [10]:
def build_model(num_classes: int = 4) -> nn.Module:
    model = timm.create_model(
        "efficientnet_b0",
        pretrained=True,
        num_classes=num_classes
    )
    return model

In [11]:
def freeze_backbone(model: nn.Module):
    """Phase 1: freeze everything except the final classifier head."""
    for name, param in model.named_parameters():
        if "classifier" in name:
            param.requires_grad = True
        else:
            param.requires_grad = False

In [12]:
def unfreeze_final_blocks(model: nn.Module, num_blocks_to_unfreeze: int = 1):
    """
    Phase 2: unfreeze EfficientNet-B0 classifier,
    head layers, and the last N feature blocks.
    """

    # Freeze everything first
    for param in model.parameters():
        param.requires_grad = False

    # Always train classifier
    for param in model.classifier.parameters():
        param.requires_grad = True

    # Also fine-tune the final EfficientNet head
    for param in model.conv_head.parameters():
        param.requires_grad = True

    for param in model.bn2.parameters():
        param.requires_grad = True

    # Unfreeze last N EfficientNet blocks
    blocks = list(model.blocks.children())

    num_blocks_to_unfreeze = min(
        num_blocks_to_unfreeze,
        len(blocks)
    )

    for block in blocks[-num_blocks_to_unfreeze:]:
        for param in block.parameters():
            param.requires_grad = True

In [13]:
def build_optimizer(model, name: str, lr: float, weight_decay: float):
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if name == "adamw":
        return torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    elif name == "adam":
        return torch.optim.Adam(trainable_params, lr=lr, weight_decay=weight_decay)
    elif name == "sgd":
        return torch.optim.SGD(trainable_params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    raise ValueError(f"Unknown optimizer: {name}")

In [14]:
def build_scheduler(optimizer, name: str, epochs: int):
    if name == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    elif name == "step":
        return torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(1, epochs // 3), gamma=0.1)
    elif name == "plateau":
        return torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=2)
    elif name == "none":
        return None
    raise ValueError(f"Unknown scheduler: {name}")

## Training / evaluation loops

In [15]:
def run_epoch(model, loader, criterion, optimizer, device, train: bool, desc: str = ""):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
 
    non_blocking = device.type == "cuda"
    context = torch.enable_grad() if train else torch.no_grad()
    progress = tqdm(loader, desc=desc, leave=False, dynamic_ncols=True)
    with context:
        for images, labels in progress:
            images = images.to(device, non_blocking=non_blocking)
            labels = labels.to(device, non_blocking=non_blocking)
            if train:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()
 
            batch_size = images.size(0)
            total_loss += loss.item() * batch_size
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += batch_size
            progress.set_postfix(loss=f"{total_loss / total:.4f}", acc=f"{correct / total:.4f}")
 
    return total_loss / total, correct / total

In [16]:
def train_phase(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    device, epochs, patience, phase_name, output_dir,
):
    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    history = []
 
    for epoch in range(1, epochs + 1):
        train_desc = f"{phase_name} epoch {epoch}/{epochs} train"
        val_desc = f"{phase_name} epoch {epoch}/{epochs} val"
        train_loss, train_acc = run_epoch(
            model, train_loader, criterion, optimizer, device, train=True, desc=train_desc
        )
        val_loss, val_acc = run_epoch(
            model, val_loader, criterion, optimizer, device, train=False, desc=val_desc
        )
 
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()
 
        history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                         "val_loss": val_loss, "val_acc": val_acc})
        print(f"[{phase_name}] epoch {epoch}/{epochs} "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")
 
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"[{phase_name}] Early stopping at epoch {epoch} (no improvement for {patience} epochs).")
                break
 
    model.load_state_dict(best_state)
    with open(Path(output_dir) / f"{phase_name}_history.json", "w") as f:
        json.dump(history, f, indent=2)
    return model

In [17]:
@torch.no_grad()
def evaluate(model, loader, class_names, device, output_dir, evaluation_type):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
 
    non_blocking = device.type == "cuda"
    for images, labels in loader:
        images = images.to(device, non_blocking=non_blocking)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)
 
        all_labels.extend(labels.numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)
 
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
 
    report = classification_report(all_labels, all_preds, target_names=class_names, digits=4, output_dict=True)
    cm = confusion_matrix(all_labels, all_preds)
 
    try:
        auc_macro = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
        auc_per_class = roc_auc_score(all_labels, all_probs, multi_class="ovr", average=None)
    except ValueError:
        auc_macro, auc_per_class = None, None
 
    results = {
        "classification_report": report,
        "confusion_matrix": cm.tolist(),
        "roc_auc_macro": auc_macro,
        "roc_auc_per_class": auc_per_class.tolist() if auc_per_class is not None else None,
        "class_names": class_names,
    }
 
    with open(Path(output_dir) / f"{evaluation_type}_test_results.json", "w") as f:
        json.dump(results, f, indent=2)
 
    print("\n=== Test set performance ===")
    print(f"Test accuracy: {accuracy_score(all_labels, all_preds):.4f}")
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))
    print("Confusion matrix:\n", cm)
    if auc_macro is not None:
        print(f"Macro ROC-AUC: {auc_macro:.4f}")

    # --- Save confusion matrix as CSV ---
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    cm_df.to_csv(Path(output_dir) / f"{evaluation_type}_confusion_matrix.csv")

    # --- Save one-row summary CSV (accuracy, macro/weighted f1 & recall, ROC-AUC) ---
    summary = {
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": report["macro avg"]["f1-score"],
        "macro_recall": report["macro avg"]["recall"],
        "macro_precision": report["macro avg"]["precision"],
        "weighted_f1": report["weighted avg"]["f1-score"],
        "weighted_recall": report["weighted avg"]["recall"],
        "roc_auc_macro": auc_macro,
    }
    pd.DataFrame([summary]).to_csv(Path(output_dir) / f"{evaluation_type}_summary_metrics.csv", index=False)
    
 
    return results

In [18]:
# ---------------- T16 experiment setup ----------------
DATA_DIR = r"/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset"
OUTPUT_ROOT = Path("./runs/T16_efficientnet_b0")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42
NUM_WORKERS = 4
UNFREEZE_BLOCKS = 1

# Short tuning runs
TUNE_PHASE1_EPOCHS = 5
TUNE_PHASE2_EPOCHS = 12
TUNE_PATIENCE = 3

# Full final run
FINAL_PHASE1_EPOCHS = 15
FINAL_PHASE2_EPOCHS = 40
FINAL_PATIENCE = 5

# Targeted sweep. We keep batch size / AdamW / cosine fixed for comparability
# and tune the most relevant learning-rate + regularization choices.
TUNING_CONFIGS = [
    {"name": "A_current",  "phase1_lr": 1e-3, "phase2_lr": 1e-5, "weight_decay": 1e-4, "optimizer": "adamw", "scheduler": "cosine"},
    {"name": "B_lr",       "phase1_lr": 3e-4, "phase2_lr": 3e-5, "weight_decay": 1e-4, "optimizer": "adamw", "scheduler": "cosine"},
    {"name": "C_lower_lr", "phase1_lr": 1e-4, "phase2_lr": 1e-5, "weight_decay": 1e-4, "optimizer": "adamw", "scheduler": "cosine"},
    {"name": "D_more_wd",  "phase1_lr": 3e-4, "phase2_lr": 3e-5, "weight_decay": 1e-3, "optimizer": "adamw", "scheduler": "cosine"},
]

pd.DataFrame(TUNING_CONFIGS)


,name,phase1_lr,phase2_lr,weight_decay,optimizer,scheduler
0,A_current,0.0010,0.00001,0.0001,adamw,cosine
1,B_lr,0.0003,0.00003,0.0001,adamw,cosine
2,C_lower_lr,0.0001,0.00001,0.0001,adamw,cosine
3,D_more_wd,0.0003,0.00003,0.0010,adamw,cosine


In [19]:
# ---------------- Device ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(device)}")
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU memory: {torch.cuda.get_device_properties(device).total_memory / (1024**3):.1f} GB")
else:
    raise RuntimeError("Enable a Kaggle GPU before running T16 tuning/final training.")


Using device: cuda
GPU: Tesla T4
CUDA: 12.8
GPU memory: 14.6 GB


In [20]:
# ---------------- Build the identical data split once ----------------
set_seed(SEED)
train_loader, val_loader, test_loader, class_names, train_targets, datasets = build_dataloaders(
    DATA_DIR, IMG_SIZE, BATCH_SIZE, SEED, NUM_WORKERS
)
num_classes = len(class_names)
print(f"Classes ({num_classes}): {class_names}")
print(f"Train/Val/Test sizes: {len(train_loader.dataset)}/{len(val_loader.dataset)}/{len(test_loader.dataset)}")

class_weights = compute_class_weights(train_targets, num_classes).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)


Classes (4): ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
Train/Val/Test sizes: 14815/3175/3175


In [21]:
def best_history_row(history_file):
    with open(history_file, "r") as f:
        history = json.load(f)
    return min(history, key=lambda x: x["val_loss"])


In [22]:
# ---------------- Validation-only targeted tuning ----------------
tuning_results = []

for cfg in TUNING_CONFIGS:
    print("\n" + "=" * 90)
    print(f"TUNING: {cfg['name']} | {cfg}")
    print("=" * 90)

    # Reset RNG so every configuration starts comparably.
    set_seed(SEED)
    cfg_dir = OUTPUT_ROOT / "tuning" / cfg["name"]
    cfg_dir.mkdir(parents=True, exist_ok=True)

    model = build_model(num_classes=num_classes).to(device)

    # Phase 1 — frozen backbone
    freeze_backbone(model)
    opt1 = build_optimizer(model, cfg["optimizer"], cfg["phase1_lr"], cfg["weight_decay"])
    sched1 = build_scheduler(opt1, cfg["scheduler"], TUNE_PHASE1_EPOCHS)
    model = train_phase(
        model, train_loader, val_loader, criterion, opt1, sched1,
        device, TUNE_PHASE1_EPOCHS, TUNE_PATIENCE,
        "phase1_frozen", cfg_dir,
    )

    # Phase 2 — fine-tune final EfficientNet block/head
    unfreeze_final_blocks(model, num_blocks_to_unfreeze=UNFREEZE_BLOCKS)
    opt2 = build_optimizer(model, cfg["optimizer"], cfg["phase2_lr"], cfg["weight_decay"])
    sched2 = build_scheduler(opt2, cfg["scheduler"], TUNE_PHASE2_EPOCHS)
    model = train_phase(
        model, train_loader, val_loader, criterion, opt2, sched2,
        device, TUNE_PHASE2_EPOCHS, TUNE_PATIENCE,
        "phase2_finetune", cfg_dir,
    )

    best = best_history_row(cfg_dir / "phase2_finetune_history.json")
    tuning_results.append({
        **cfg,
        "best_epoch": best["epoch"],
        "best_val_loss": best["val_loss"],
        "val_acc_at_best_loss": best["val_acc"],
    })

    del model, opt1, opt2, sched1, sched2
    torch.cuda.empty_cache()

tuning_df = pd.DataFrame(tuning_results).sort_values("best_val_loss").reset_index(drop=True)
tuning_df.to_csv(OUTPUT_ROOT / "tuning_summary.csv", index=False)
print("\n=== TUNING SUMMARY: lower validation loss is better ===")
display(tuning_df)



TUNING: A_current | {'name': 'A_current', 'phase1_lr': 0.001, 'phase2_lr': 1e-05, 'weight_decay': 0.0001, 'optimizer': 'adamw', 'scheduler': 'cosine'}


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

[phase1_frozen] epoch 1/5 train_loss=1.5642 train_acc=0.5249 val_loss=0.9713 val_acc=0.6661


[phase1_frozen] epoch 2/5 train_loss=0.9129 train_acc=0.6716 val_loss=0.8025 val_acc=0.7140


[phase1_frozen] epoch 3/5 train_loss=0.7659 train_acc=0.7118 val_loss=0.7143 val_acc=0.7458


...

[phase2_finetune] epoch 9/12 train_loss=0.2960 train_acc=0.8660 val_loss=0.3176 val_acc=0.8746


[phase2_finetune] epoch 10/12 train_loss=0.3105 train_acc=0.8598 val_loss=0.3204 val_acc=0.8759


[phase2_finetune] epoch 11/12 train_loss=0.3048 train_acc=0.8665 val_loss=0.3113 val_acc=0.8784


[phase2_finetune] epoch 12/12 train_loss=0.2890 train_acc=0.8674 val_loss=0.3141 val_acc=0.8750

=== TUNING SUMMARY: lower validation loss is better ===


,name,phase1_lr,phase2_lr,weight_decay,optimizer,scheduler,best_epoch,best_val_loss,val_acc_at_best_loss
0,D_more_wd,0.0003,0.00003,0.0010,adamw,cosine,11,0.311329,0.878425
1,B_lr,0.0003,0.00003,0.0001,adamw,cosine,11,0.311398,0.878425
2,A_current,0.0010,0.00001,0.0001,adamw,cosine,11,0.409269,0.840315
3,C_lower_lr,0.0001,0.00001,0.0001,adamw,cosine,11,0.480535,0.822047


In [23]:
# ---------------- Select tuning winner ----------------
winner_name = tuning_df.iloc[0]["name"]
winner = next(cfg for cfg in TUNING_CONFIGS if cfg["name"] == winner_name)

with open(OUTPUT_ROOT / "selected_config.json", "w") as f:
    json.dump(winner, f, indent=2)

print("Selected configuration:")
print(json.dumps(winner, indent=2))


Selected configuration:
{
  "name": "D_more_wd",
  "phase1_lr": 0.0003,
  "phase2_lr": 3e-05,
  "weight_decay": 0.001,
  "optimizer": "adamw",
  "scheduler": "cosine"
}


In [24]:
# ---------------- Full final run using the validation-selected configuration ----------------
FINAL_DIR = OUTPUT_ROOT / "final_selected"
FINAL_DIR.mkdir(parents=True, exist_ok=True)
set_seed(SEED)

final_model = build_model(num_classes=num_classes).to(device)

# Full Phase 1
freeze_backbone(final_model)
opt1 = build_optimizer(final_model, winner["optimizer"], winner["phase1_lr"], winner["weight_decay"])
sched1 = build_scheduler(opt1, winner["scheduler"], FINAL_PHASE1_EPOCHS)
final_model = train_phase(
    final_model, train_loader, val_loader, criterion, opt1, sched1,
    device, FINAL_PHASE1_EPOCHS, FINAL_PATIENCE,
    "phase1_frozen", FINAL_DIR,
)

# Preserve the best frozen model, but DO NOT touch the test set yet.
frozen_state = copy.deepcopy(final_model.state_dict())
torch.save({
    "model_state_dict": frozen_state,
    "class_names": class_names,
    "selected_config": winner,
}, FINAL_DIR / "efficientnet_b0_T16_frozen.pt")

# Full Phase 2
unfreeze_final_blocks(final_model, num_blocks_to_unfreeze=UNFREEZE_BLOCKS)
opt2 = build_optimizer(final_model, winner["optimizer"], winner["phase2_lr"], winner["weight_decay"])
sched2 = build_scheduler(opt2, winner["scheduler"], FINAL_PHASE2_EPOCHS)
final_model = train_phase(
    final_model, train_loader, val_loader, criterion, opt2, sched2,
    device, FINAL_PHASE2_EPOCHS, FINAL_PATIENCE,
    "phase2_finetune", FINAL_DIR,
)

torch.save({
    "model_state_dict": final_model.state_dict(),
    "class_names": class_names,
    "selected_config": winner,
}, FINAL_DIR / "efficientnet_b0_T16_finetuned.pt")

print("Final training complete. Test set has not been used during tuning/model selection.")


[phase1_frozen] epoch 1/15 train_loss=2.2898 train_acc=0.3871 val_loss=1.5907 val_acc=0.5087


[phase1_frozen] epoch 2/15 train_loss=1.4024 train_acc=0.5471 val_loss=1.2344 val_acc=0.5893


[phase1_frozen] epoch 3/15 train_loss=1.1370 train_acc=0.6075 val_loss=1.0210 val_acc=0.6545


[phase1_frozen] epoch 4/15 train_loss=0.9843 train_acc=0.6422 val_loss=0.9733 val_acc=0.6624


[phase1_frozen] epoch 5/15 train_loss=0.9143 train_acc=0.6611 val_loss=0.8724 val_acc=0.7030


...

[phase2_finetune] epoch 24/40 train_loss=0.1807 train_acc=0.9152 val_loss=0.2376 val_acc=0.9083


[phase2_finetune] epoch 25/40 train_loss=0.1765 train_acc=0.9137 val_loss=0.2534 val_acc=0.9046


[phase2_finetune] epoch 26/40 train_loss=0.1779 train_acc=0.9157 val_loss=0.2398 val_acc=0.9065


[phase2_finetune] epoch 27/40 train_loss=0.1781 train_acc=0.9154 val_loss=0.2439 val_acc=0.9071


[phase2_finetune] epoch 28/40 train_loss=0.1699 train_acc=0.9179 val_loss=0.2386 val_acc=0.9065
[phase2_finetune] Early stopping at epoch 28 (no improvement for 5 epochs).
Final training complete. Test set has not been used during tuning/model selection.


In [25]:
# ---------------- Final held-out test evaluation ----------------
# Frozen result
frozen_model = build_model(num_classes=num_classes).to(device)
frozen_model.load_state_dict(frozen_state)
frozen_results = evaluate(
    frozen_model, test_loader, class_names, device, FINAL_DIR, "frozen"
)

# Fine-tuned result
finetuned_results = evaluate(
    final_model, test_loader, class_names, device, FINAL_DIR, "finetuned"
)

del frozen_model
torch.cuda.empty_cache()



=== Test set performance ===
Test accuracy: 0.7408
                 precision    recall  f1-score   support

          COVID     0.5808    0.7159    0.6413       542
   Lung_Opacity     0.7151    0.7373    0.7260       902
         Normal     0.8346    0.7390    0.7839      1529
Viral Pneumonia     0.7578    0.8366    0.7953       202

       accuracy                         0.7408      3175
      macro avg     0.7221    0.7572    0.7366      3175
   weighted avg     0.7524    0.7408    0.7438      3175

Confusion matrix:
 [[ 388   73   72    9]
 [ 106  665  127    4]
 [ 170  188 1130   41]
 [   4    4   25  169]]
Macro ROC-AUC: 0.9151

=== Test set performance ===
Test accuracy: 0.9087
                 precision    recall  f1-score   support

          COVID     0.9044    0.9428    0.9232       542
   Lung_Opacity     0.8850    0.8792    0.8821       902
         Normal     0.9211    0.9091    0.9151      1529
Viral Pneumonia     0.9317    0.9455    0.9386       202

       accuracy 

In [26]:
# ---------------- T16 final summary ----------------
frozen_summary = pd.read_csv(FINAL_DIR / "frozen_summary_metrics.csv").iloc[0].to_dict()
finetuned_summary = pd.read_csv(FINAL_DIR / "finetuned_summary_metrics.csv").iloc[0].to_dict()

final_summary = pd.DataFrame([
    {"stage": "Frozen", **frozen_summary},
    {"stage": "Fine-tuned", **finetuned_summary},
])
final_summary.to_csv(OUTPUT_ROOT / "T16_final_summary.csv", index=False)

print("Selected config:", winner)
display(final_summary)
print("\nT16 COMPLETE")
print(f"Keep/download this folder: {OUTPUT_ROOT}")


Selected config: {'name': 'D_more_wd', 'phase1_lr': 0.0003, 'phase2_lr': 3e-05, 'weight_decay': 0.001, 'optimizer': 'adamw', 'scheduler': 'cosine'}


,stage,accuracy,macro_f1,macro_recall,macro_precision,weighted_f1,weighted_recall,roc_auc_macro
0,Frozen,0.740787,0.736626,0.757199,0.722076,0.743834,0.740787,0.915100
1,Fine-tuned,0.908661,0.914739,0.919149,0.910579,0.908590,0.908661,0.984134



T16 COMPLETE
Keep/download this folder: runs/T16_efficientnet_b0
